# v7 - Corte profundo de NEV + threshold conservador para MEL

Evolucao do v6. Mantem o cache de embeddings (encoder congelado) e adiciona:

**1. Corte mais agressivo de NEV (`NEV_TARGET`)**
- v6 cortava 2000 NEV (sobravam ~3700). Aqui o NEV do HAM eh reduzido a `NEV_TARGET` imagens.
- Objetivo: reduzir o vies NEV-> e converter parte dos MEL->NEV (falsos negativos clinicos) em acertos.
- Trade-off honesto: como o test do Derm7pt eh ~55% NEV, a accuracy global pode cair um pouco,
  mas macro-F1 e MEL recall (objetivo clinico) tendem a subir. A selecao por macro-F1 escolhe o equilibrio.

**2. Threshold conservador para MEL (regra de decisao, nao de treino)**
- Apos o softmax: prediz MEL se P(MEL) >= tau, mesmo que nao seja o argmax.
- 'Conservador' = tau baixo => pega mais melanomas ao custo de mais falsos positivos NEV.
- O tau eh CALIBRADO no Derm7pt val (nunca no test) e so depois aplicado ao test.

Tudo o mais (arquitetura, LR, batch, FocalLoss, inverse_sqrt, early stopping) inalterado vs v6.

In [ ]:
import os
import sys
import site
import importlib
import subprocess

REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "huggingface_hub"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
site.main()

print("Setup OK")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter
from sklearn.metrics import f1_score, recall_score, classification_report
from kaggle_secrets import UserSecretsClient

from melanoma_tcc.data.preprocessing import (
    Derm7ptUnifiedDataset, HAM10000Dataset, CombinedDermDataset,
    classification_collate_fn,
    GROUP_TO_LABEL, LABEL_TO_GROUP, METADATA_DIM_V5,
    HAM_DX_TO_GROUP, ham10000_train_val_split,
)
from melanoma_tcc.model.classifier import build_dermclassifier
from melanoma_tcc.model.losses import FocalLoss, compute_class_weights
from melanoma_tcc.utils.metrics import compute_metrics, plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('mellanoma_TCC')

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7-pt-dataset/release_v0"
DERM_META = f"{DERM7PT_DIR}/meta/meta.csv"
DERM_IMAGES = f"{DERM7PT_DIR}/images"
DERM_TRAIN_IDX = f"{DERM7PT_DIR}/meta/train_indexes.csv"
DERM_VAL_IDX = f"{DERM7PT_DIR}/meta/valid_indexes.csv"
DERM_TEST_IDX = f"{DERM7PT_DIR}/meta/test_indexes.csv"

HAM_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
HAM_META = f"{HAM_DIR}/HAM10000_metadata.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEL_LABEL = GROUP_TO_LABEL['MEL']
print(f"Device: {device}")
print(f"Derm7pt meta existe: {os.path.exists(DERM_META)}")
print(f"HAM meta existe: {os.path.exists(HAM_META)}")
print(f"Metadata dim v5: {METADATA_DIM_V5} | MEL label = {MEL_LABEL}")

In [ ]:
model, processor = build_dermclassifier(
    hf_token=HF_TOKEN,
    num_classes=5,
    metadata_dim=METADATA_DIM_V5,
    freeze_vision=True,
)
model = model.to(device)
model.vision_encoder = model.vision_encoder.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ===== PARAMETRO DE BALANCEAMENTO =====
NEV_TARGET = 1400   # imagens NEV mantidas no HAM train (v6 deixava ~3700). Varie 1000/1400/1800.

# Derm7pt: schema unificado. augment=False em TODOS (cache de embedding exige imagem fixa).
derm_train = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                   indexes_csv=DERM_TRAIN_IDX, augment=False, seed=42)
derm_val = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                 indexes_csv=DERM_VAL_IDX, augment=False)
derm_test = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                  indexes_csv=DERM_TEST_IDX, augment=False)

# HAM10000: filtra unknowns + split por lesion_id
ham_train_df, ham_val_df = ham10000_train_val_split(HAM_META, val_ratio=0.15, seed=42,
                                                    filter_unknown=True)
ham_train_df = ham_train_df.copy()
ham_train_df['group'] = (ham_train_df['dx'].str.strip().str.lower()
                         .map(HAM_DX_TO_GROUP).fillna('MISC'))

def _train_dist(derm_ds, ham_df):
    return Counter(pd.concat([derm_ds.df['group'], ham_df['group']]))

print("Distribuicao TRAIN ANTES do corte:")
print(_train_dist(derm_train, ham_train_df))

# ---- CORTE PROFUNDO: reduz NEV do HAM a NEV_TARGET ----
nev_mask = ham_train_df['group'] == 'NEV'
n_nev = int(nev_mask.sum())
if n_nev > NEV_TARGET:
    keep_nev = ham_train_df[nev_mask].sample(n=NEV_TARGET, random_state=42)
    ham_train_df = pd.concat([keep_nev, ham_train_df[~nev_mask]]).reset_index(drop=True)
    print(f"\nHAM NEV reduzido de {n_nev} -> {NEV_TARGET}")
else:
    print(f"\nHAM NEV ({n_nev}) <= NEV_TARGET ({NEV_TARGET}); sem corte.")
# -------------------------------------------------------

print("\nDistribuicao TRAIN DEPOIS do corte:")
print(_train_dist(derm_train, ham_train_df))

ham_train = HAM10000Dataset(ham_train_df, HAM_DIR, processor, augment=False, seed=42)
ham_val = HAM10000Dataset(ham_val_df, HAM_DIR, processor, augment=False)

train_dataset = CombinedDermDataset([derm_train, ham_train])
print(f"\nTrain combinado: {len(train_dataset)} = derm({len(derm_train)}) + ham({len(ham_train)})")
print(f"Derm val: {len(derm_val)} | HAM val: {len(ham_val)} | Derm test: {len(derm_test)}")

In [ ]:
BATCH_SIZE = 32
EPOCHS = 40          # teto de seguranca; early stopping decide o fim real
PATIENCE = 6         # epocas sem melhora de macro-F1 (derm_val) -> para
LR = 5e-4
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05
CONSERVATIVE_TAU = 0.30   # limiar inicial de MEL (calibrado no val abaixo)

# Class weights a partir do train JA cortado (inverse_sqrt mantido do v6)
train_groups = pd.concat([derm_train.df['group'], ham_train.df['group']])
group_counts = Counter(train_groups)
class_counts = [group_counts.get(LABEL_TO_GROUP[i], 1) for i in range(5)]
alpha = compute_class_weights(class_counts, mode="inverse_sqrt")
print(f"Class counts: {[(LABEL_TO_GROUP[i], class_counts[i]) for i in range(5)]}")
print(f"Class weights (alpha, mode=inverse_sqrt): {alpha.tolist()}")

criterion = FocalLoss(alpha=alpha, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

In [ ]:
# ===== PRE-COMPUTA EMBEDDINGS (unica vez que o encoder roda) =====
@torch.no_grad()
def precompute_embeddings(model, dataset, batch_size=32):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=classification_collate_fn,
                        num_workers=4, pin_memory=True)
    embs, metas, labels = [], [], []
    for i, batch in enumerate(loader):
        pooled = model.encode_image(batch['pixel_values'].to(device))
        embs.append(pooled.cpu())
        metas.append(batch['metadata'])
        labels.append(batch['labels'])
        if i % 20 == 0:
            print(f"  batch {i}/{len(loader)}")
    return TensorDataset(torch.cat(embs), torch.cat(metas), torch.cat(labels))

print("Cacheando train...")
train_cached = precompute_embeddings(model, train_dataset, BATCH_SIZE)
print("Cacheando derm_val...")
derm_val_cached = precompute_embeddings(model, derm_val, BATCH_SIZE)
print("Cacheando ham_val...")
ham_val_cached = precompute_embeddings(model, ham_val, BATCH_SIZE)
print("Cacheando derm_test...")
derm_test_cached = precompute_embeddings(model, derm_test, BATCH_SIZE)

emb_dim = train_cached.tensors[0].shape[1]
print(f"\nEmbedding dim: {emb_dim} | train N={len(train_cached)}")

train_loader = DataLoader(train_cached, batch_size=BATCH_SIZE, shuffle=True)
derm_val_loader = DataLoader(derm_val_cached, batch_size=BATCH_SIZE, shuffle=False)
ham_val_loader = DataLoader(ham_val_cached, batch_size=BATCH_SIZE, shuffle=False)
derm_test_loader = DataLoader(derm_test_cached, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# ===== Treino so da cabeca a partir dos embeddings cacheados =====
def head_forward(model, emb, md):
    v_feat = model.vision_proj(emb.to(device))
    m_feat = model.metadata_encoder(md.float().to(device))
    return model.classifier(torch.cat([v_feat, m_feat], dim=-1))

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for emb, md, lb in loader:
            lb = lb.to(device)
            logits = head_forward(model, emb, md)
            loss = criterion(logits, lb)
            total_loss += loss.item() * lb.size(0)
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(lb.cpu().tolist())
    n = len(loader.dataset)
    avg_loss = total_loss / n
    acc = sum(int(p == l) for p, l in zip(all_preds, all_labels)) / n
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, macro_f1, all_preds, all_labels

best_val_f1 = 0.0
best_state = None
best_epoch = 0
epochs_no_improve = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    model.vision_encoder.eval()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0
    for emb, md, lb in train_loader:
        lb = lb.to(device)
        logits = head_forward(model, emb, md)
        loss = criterion(logits, lb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        train_loss_sum += loss.item() * lb.size(0)
        train_correct += (logits.argmax(dim=-1) == lb).sum().item()
        train_total += lb.size(0)
    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    val_loss, val_acc, val_f1, _, _ = evaluate(model, derm_val_loader, criterion)
    scheduler.step(val_f1)
    cur_lr = optimizer.param_groups[0]['lr']
    history.append((epoch, train_loss, train_acc, val_loss, val_acc, val_f1, cur_lr))
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.3f} macroF1={val_f1:.4f} | lr={cur_lr:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items() if 'vision_encoder' not in k}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping na epoch {epoch} (sem melhora ha {PATIENCE} epocas).")
            break

print(f"\nBest macro-F1 (derm_val): {best_val_f1:.4f} na epoch {best_epoch}")

In [ ]:
if best_state is not None:
    current = model.state_dict()
    current.update(best_state)
    model.load_state_dict(current, strict=False)
    print("Best model carregado.")

os.makedirs("/kaggle/working/derm-classifier-v7", exist_ok=True)
torch.save(best_state, "/kaggle/working/derm-classifier-v7/classifier_head.pt")
print("Salvou /kaggle/working/derm-classifier-v7/classifier_head.pt")

import json
with open("/kaggle/working/derm-classifier-v7/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# ===== CALIBRACAO DO THRESHOLD DE MEL (somente no DERM_VAL) =====
@torch.no_grad()
def predict_with_mel_threshold(model, loader, tau):
    """Prediz MEL se P(MEL) >= tau; caso contrario usa o argmax."""
    model.eval()
    preds, labels = [], []
    for emb, md, lb in loader:
        probs = F.softmax(head_forward(model, emb, md), dim=-1)
        argmax_pred = probs.argmax(dim=-1)
        mel_trigger = probs[:, MEL_LABEL] >= tau
        pred = torch.where(mel_trigger, torch.full_like(argmax_pred, MEL_LABEL), argmax_pred)
        preds.extend(pred.cpu().tolist())
        labels.extend(lb.tolist())
    return labels, preds

print("Sweep de tau no DERM_VAL (calibracao - nao toca no test):")
print(f"{'tau':>6} | {'MEL_recall':>10} | {'MEL_prec':>9} | {'macroF1':>8} | {'acc':>6}")
for tau in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]:
    yv, pv = predict_with_mel_threshold(model, derm_val_loader, tau)
    mel_rec = recall_score(yv, pv, labels=[MEL_LABEL], average='macro', zero_division=0)
    mel_prec = (sum(1 for y, p in zip(yv, pv) if p == MEL_LABEL and y == MEL_LABEL)
                / max(1, sum(1 for p in pv if p == MEL_LABEL)))
    macro = f1_score(yv, pv, average='macro')
    acc = sum(int(y == p) for y, p in zip(yv, pv)) / len(yv)
    print(f"{tau:>6.2f} | {mel_rec:>10.3f} | {mel_prec:>9.3f} | {macro:>8.4f} | {acc:>6.3f}")

print(f"\nTAU escolhido (conservador inicial): {CONSERVATIVE_TAU}")
print("Ajuste CONSERVATIVE_TAU acima conforme o sweep e re-rode as celulas de avaliacao.")

In [ ]:
# ===== AVALIACAO FINAL: argmax (baseline) vs threshold conservador =====
TARGET_NAMES = ["BCC", "NEV", "MEL", "SK", "MISC"]

def report_block(titulo, loader, cm_path):
    print("=" * 64)
    print(titulo)
    print("=" * 64)
    # argmax puro
    _, acc, mf1, preds, labels = evaluate(model, loader, criterion)
    print(f"[ARGMAX]   accuracy={acc:.4f} | macro-F1={mf1:.4f}")
    print(classification_report(labels, preds, target_names=TARGET_NAMES, digits=4, zero_division=0))
    # threshold conservador de MEL
    yt, pt = predict_with_mel_threshold(model, loader, CONSERVATIVE_TAU)
    acc_t = sum(int(y == p) for y, p in zip(yt, pt)) / len(yt)
    mf1_t = f1_score(yt, pt, average='macro')
    print(f"[TAU={CONSERVATIVE_TAU}]  accuracy={acc_t:.4f} | macro-F1={mf1_t:.4f}")
    print(classification_report(yt, pt, target_names=TARGET_NAMES, digits=4, zero_division=0))
    plot_confusion_matrix(yt, pt, save_path=cm_path, target_names=TARGET_NAMES)
    return (labels, preds), (yt, pt)

derm_argmax, derm_thr = report_block(
    "DOMINIO 1: Derm7pt TEST (comparacao com v3/v4/v6)",
    derm_test_loader, '/kaggle/working/derm-classifier-v7-derm-cm.png')
print()
ham_argmax, ham_thr = report_block(
    "DOMINIO 2: HAM TEST (ham_val, nao usado na selecao)",
    ham_val_loader, '/kaggle/working/derm-classifier-v7-ham-cm.png')

In [ ]:
# Salva predicoes (argmax e threshold) dos dois dominios
def save_preds(path, argmax_tuple, thr_tuple):
    labels, preds = argmax_tuple
    _, preds_thr = thr_tuple
    df = pd.DataFrame({
        'true_label': labels,
        'pred_argmax': preds,
        'pred_threshold': preds_thr,
        'true_group': [LABEL_TO_GROUP[l] for l in labels],
        'pred_group_argmax': [LABEL_TO_GROUP[p] for p in preds],
        'pred_group_threshold': [LABEL_TO_GROUP[p] for p in preds_thr],
    })
    df.to_csv(path, index=False)
    return df

derm_pred_df = save_preds('/kaggle/working/derm_classifier_v7_derm_predictions.csv',
                          derm_argmax, derm_thr)
ham_pred_df = save_preds('/kaggle/working/derm_classifier_v7_ham_predictions.csv',
                         ham_argmax, ham_thr)

print(f"Derm7pt test: {len(derm_pred_df)} predicoes salvas.")
print(f"  argmax dist:    {Counter(derm_pred_df['pred_group_argmax'])}")
print(f"  threshold dist: {Counter(derm_pred_df['pred_group_threshold'])}")
print(f"  true dist:      {Counter(derm_pred_df['true_group'])}")
print(f"\nHAM test: {len(ham_pred_df)} predicoes salvas.")
print(f"  argmax dist:    {Counter(ham_pred_df['pred_group_argmax'])}")
print(f"  threshold dist: {Counter(ham_pred_df['pred_group_threshold'])}")
print(f"  true dist:      {Counter(ham_pred_df['true_group'])}")